# Data Extraction Flow — Climate (MERGE CPTEC + NASA POWER)

| branch | module | obs |
|---|---|---|
| MERGE CPTEC | `pipelines.merge` | *while loop* até `has_more=False`; `max_days=120`, `lookback_days=7` |
| NASA POWER | `pipelines.power` | *while loop* até `has_more=False`; `max_days=366`, `lookback_days=75` |

If `dim_municipiosIbge` is not in the lake yet, the flow runs `pipelines.ibge` first.

In [ ]:
import pandas as pd
from pipelines.flow import run_flow
from lakehouse import core as C

pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 60)

## Inputs

In [ ]:
mode = "incremental"      # incremental | range | backfill
reference_date = ""         # AAAA-MM-DD; vazio = hoje (America/Sao_Paulo)
start_date = ""             # AAAA-MM-DD (obrigatório em range)
end_date = ""
force = False               # MERGE: rebaixa dias já carregados
target_db = "estudo"        # pasta de destino no bucket lake (lake/estudo/)
ref_db = ""                 # onde está dim_municipiosIbge; vazio = target_db
power_parametros = ["PRECTOTCORR"]  # PRECTOTCORR, T2M, T2M_MAX, T2M_MIN, RH2M, WS2M
dry_run = True              # True = só plano + verificação leve das fontes

## Execute

In [ ]:
spark = None if dry_run else C.get_spark("flow_clima", cores_max=4)
resumo = run_flow(spark, mode=mode, reference_date=reference_date, start_date=start_date, end_date=end_date, force=force, target_db=target_db, ref_db=ref_db, power_parametros=power_parametros, dry_run=dry_run)
pd.DataFrame(resumo)

## Bucket usage

The bucket has a hard quota (5 GiB by default); every pipeline estimates its write and refuses one that would not fit.

In [ ]:
C.quota_check(0)